In [54]:
import os
from dotenv import load_dotenv
load_dotenv(override=True) 

AI_KEY = os.environ['AI_KEY']
AI_MODEL_NAME = os.environ['AI_MODEL_NAME']

from google import genai

client = genai.Client(api_key=AI_KEY)

python-dotenv could not parse statement starting at line 6
python-dotenv could not parse statement starting at line 8
python-dotenv could not parse statement starting at line 9
python-dotenv could not parse statement starting at line 11
python-dotenv could not parse statement starting at line 15
python-dotenv could not parse statement starting at line 16
python-dotenv could not parse statement starting at line 17
python-dotenv could not parse statement starting at line 18
python-dotenv could not parse statement starting at line 19
python-dotenv could not parse statement starting at line 20
python-dotenv could not parse statement starting at line 21
python-dotenv could not parse statement starting at line 23
python-dotenv could not parse statement starting at line 26


In [55]:
AI_JSON_RESPONSE_TYPE = {"response_mime_type": "application/json"}
def query_ai(generation_config, prompt,):
    try:
        response = client.models.generate_content(
            model=AI_MODEL_NAME,
            config=generation_config,
            contents=prompt
        )
        return response.text
    except Exception as e:
        print('query_ai Exception', e)
        return ''

<h2>Read CV</h2>

In [56]:
TEST_PDF_FILEPATH = 'samples/cv1.pdf'

In [57]:
import docx
from PyPDF2 import PdfReader

def extract_text_from_file(file_path):
    if file_path.endswith(".docx"):
        doc = docx.Document(file_path)
        return "\n".join([p.text for p in doc.paragraphs])

    elif file_path.endswith(".pdf"):
        reader = PdfReader(file_path)
        return "".join([page.extract_text() or "" for page in reader.pages])

    else:
        raise ValueError("Unsupported file format")

#
raw_cv_content = extract_text_from_file(TEST_PDF_FILEPATH)
# print(raw_cv_content)

<h2>Evaluate CV</h2>

In [58]:
TEST_JOB_DETAILS = """
Full job description
Job description
Ahamove is looking for a Junior Backend Software Engineer to join our engineering team and help build scalable systems that process hundreds of thousands of orders every day. You will work in a fast-moving environment with challenging problems in performance, reliability, distributed systems, and real-time data processing.
This role is ideal for those who are eager to learn, grow quickly, and work with large-scale backend architectures and cloud infrastructure.
Key Responsibilities
Develop, maintain, and optimize backend services (microservices).
Build and integrate APIs for customers, drivers, partners, and internal systems.
Participate in designing system architecture, data models, and processing flows.
Write clean, maintainable, well-structured code following clean architecture principles.
Troubleshoot issues, fix bugs, and improve service performance.
Collaborate closely with Product Managers, QA, Data teams, and other engineers.
Participate in code reviews and continuously improve team coding standards.
Job requirement
1-2 years of experience as a Backend Engineer.
Bachelor’s degree in Computer Science, Software Engineering, or related fields.
Proficiency in Python or Golang (at least one required).
Solid understanding of backend fundamentals: REST APIs, databases, HTTP, data structures, algorithms.
Basic SQL knowledge and experience with MongoDB or PostgreSQL
Strong problem-solving skills, growth mindset, and high ownership.
Nice-to-have
Experience with Node.js or JavaScript.
Understanding of microservices architecture.
Experience working with Kubernetes (K8s) or similar container orchestration systems.
Experience with AWS or other cloud platforms.
Familiarity with Docker, CI/CD, or message queues (Kafka, RabbitMQ).
Exposure to clean architecture or domain-driven design (DDD).
Benefit
Physical Wellbeing Benefit: General Insurance, Medical check-up, Accident Insurance, Healthcare Insurance
Emotional Wellbeing Benefit: Company Trip, Year End Party, Aha Hour Activities, Special Day Gifts, Aha Club (Badminton, Soccer)
Financial Wellbeing Benefit: Grab/Be For Work, Workplace Relocation, 13th Month Salary, PP Appreciate, Annual Leave Remain, ESOP
"""

In [ ]:
def evaluate_cv_with_ai(cv_text, job_text):
    prompt = f"""
        You are an ATS (Applicant Tracking System) and career expert.

        Evaluate this CV against the job description.

        Return JSON format ONLY:

        {{
        "score": number (0-100),
        "keyword_match": number (0-100),
        "experience_match": number (0-100),
        "skills_missing": [list],
        "strengths": [list],
        "weaknesses": [list],
        "suggestions": [list of actionable improvements]
        }}
        STRICT RULES:
      - Output must be valid JSON
      - No explanation
      - No markdown
      - No backticks

        Job Description:
        {job_text}

        Candidate CV:
        {cv_text}
        """

    response = query_ai(AI_JSON_RESPONSE_TYPE, prompt)
    return response
#test
# evaluations = evaluate_cv_with_ai(raw_cv_content, TEST_JOB_DETAILS) #approx. 28 seconds
# print(evaluations)

<h2>Rewrite CV</h2>

In [ ]:
def rewrite_structured(cv_text, job_text):
    prompt = f"""
      Rewrite the CV into structured JSON format.

      Return ONLY JSON:

      {{
        "name": "",
        "email": "",
        "phone": "",
        "location": "",
        "summary": "",
        "skills": [],
        "experience": [
          {{
            "title": "",
            "company": "",
            "start": "",
            "end": "",
            "bullets": []
          }}
        ],
        "education": [
          {{
            "degree": "",
            "school": "",
            "year": ""
          }}
        ]
      }}

      Optimize for ATS and job description.
      STRICT RULES:
      - Output must be valid JSON
      - No explanation
      - No markdown
      - No backticks

      Job:
      {job_text}

      CV:
      {cv_text}
      """

    response = query_ai(AI_JSON_RESPONSE_TYPE, prompt)

    return response
#
rewrite_text = rewrite_structured(raw_cv_content, TEST_JOB_DETAILS) #approx. 20 seconds

In [62]:
import json

def safe_parse(text):
    try:
        return json.loads(text)
    except:
        return {"error": "Invalid JSON", "raw": text}

#
new_rewrite_text = json.loads(rewrite_text) #new CV content



<h2>Rewrite CV</h2>

In [63]:
CV_TEMPLATE_PATH = 'templates/classic.html'

CV_HTML_OUTPUT_PATH = 'outputs/cv1.html'
CV_PDF_OUTPUT_PATH = 'outputs/cv1.pdf'

In [64]:
from jinja2 import Template

def render_cv(data):
    with open(CV_TEMPLATE_PATH) as f:
        template = Template(f.read())

    html = template.render(**data)

    with open(CV_HTML_OUTPUT_PATH, "w") as f:
        f.write(html)

    return html
#

new_html = render_cv(new_rewrite_text)

In [ ]:
from weasyprint import HTML

HTML(CV_HTML_OUTPUT_PATH).write_pdf(CV_PDF_OUTPUT_PATH) #approx. 2s